# Gemma 4 with reasoning disabled

An additional include-news run on the same 559 curated emails. Only the reasoning request setting changes: `reasoning: {"enabled": false}`. The prompt still asks for the same short classification reason. “System 1” is an informal description of this setting, not a measured cognitive property.

The historical Gemma run omitted the reasoning parameter; its reasoning status is unknown. This is a comparison against provider-default behavior, not a controlled reasoning-on/off experiment. Provider/version changes may also affect results.

[OpenRouter reasoning documentation](https://openrouter.ai/docs/guides/best-practices/reasoning-tokens). `exclude: true` only hides reasoning; it does not disable it. Missing token telemetry is unknown, not zero.


In [1]:
import csv
import hashlib
import json
import os
import sys
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/gemma4_openrouter.py').exists())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from scripts.gemma4_openrouter import MODEL, MAX_EMAIL_CHARS, config_hash, load_labeled_rows, run_classifier, evaluate
DATA = REPO / 'data'
OUTPUT = DATA / 'gemma4_include_no_reasoning_test_predictions.csv'

def read_csv(path):
    with path.open(newline='', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))

selected_rows = load_labeled_rows(DATA / 'enron_labeled_curated_test.csv')
baseline = read_csv(DATA / 'gemma4_include_test_predictions.csv')
assert len(selected_rows) == len(baseline) == 559
baseline_by_row = {int(r['source_row']): r for r in baseline}
assert len(baseline_by_row) == len(baseline)
assert set(baseline_by_row) == set(range(len(selected_rows)))
# Reconstruct historical CRLF input only where its exact saved hash confirms it.
# This leaves the source CSV untouched and uses identical model input bytes.
for row in selected_rows:
    old = baseline_by_row[row['source_row']]
    assert int(old['label']) == row['label']
    assert old['config_sha256'] == config_hash(news_policy='include')
    candidates = [row['email'], row['email'].replace('\r\n', '\n').replace('\n', '\r\n')]
    matching = [e for e in candidates if hashlib.sha256(e.encode()).hexdigest() == old['email_sha256']]
    if not matching:
        raise ValueError(f"Historical input mismatch at row {row['source_row']}")
    row['email'] = matching[0]
print('Verified historical inputs:', len(selected_rows), '| model:', MODEL)


Verified historical inputs: 559 | model: google/gemma-4-26b-a4b-it


In [2]:
# A one-row pilot checks provider acceptance before the remaining requests.
api_key = os.environ.get('OPENROUTER_API_KEY', '')
pilot = run_classifier(selected_rows[:1], OUTPUT, api_key, news_policy='include', reasoning_enabled=False, max_workers=1)
print('Pilot reasoning tokens:', pilot[0].get('reasoning_tokens', 'unknown'))
if float(pilot[0].get('reasoning_tokens') or 0) > 0 or str(pilot[0].get('reasoning_returned')).lower() == 'true':
    raise RuntimeError('Provider reported reasoning despite the disable request; inspect before continuing')
results = run_classifier(selected_rows, OUTPUT, api_key, news_policy='include', reasoning_enabled=False, max_workers=10)


Pilot reasoning tokens: 0
Completed 11/559 selected rows
Completed 21/559 selected rows
Completed 31/559 selected rows
Completed 41/559 selected rows
Completed 51/559 selected rows
Completed 61/559 selected rows
Completed 71/559 selected rows
Completed 81/559 selected rows
Completed 91/559 selected rows
Completed 101/559 selected rows
Completed 111/559 selected rows
Completed 121/559 selected rows
Completed 131/559 selected rows
Completed 141/559 selected rows
Completed 151/559 selected rows
Completed 161/559 selected rows
Completed 171/559 selected rows
Completed 181/559 selected rows
Completed 191/559 selected rows
Completed 201/559 selected rows
Completed 211/559 selected rows
Completed 221/559 selected rows
Completed 231/559 selected rows
Completed 241/559 selected rows
Completed 251/559 selected rows
Completed 261/559 selected rows
Completed 271/559 selected rows
Completed 281/559 selected rows
Completed 291/559 selected rows
Completed 301/559 selected rows
Completed 311/559 selec

In [3]:
metrics = evaluate(results)
reported = [r for r in results if r.get('reasoning_tokens') not in ('', None)]
violations = [int(r['source_row']) for r in results if float(r.get('reasoning_tokens') or 0) > 0 or str(r.get('reasoning_returned')).lower() == 'true']
summary = {'model': MODEL, 'news_policy': 'include', 'reasoning': {'enabled': False},
           'metrics': metrics, 'reasoning_telemetry_rows': len(reported),
           'reasoning_violation_rows': violations,
           'baseline_reasoning': 'unspecified; historical reasoning use unknown'}
(DATA / 'gemma4_include_no_reasoning_evaluation.json').write_text(json.dumps(summary, indent=2) + '\n')
print('Reasoning token telemetry:', len(reported), '/', len(results))
print('Rows reporting reasoning despite disable request:', violations)
print(f"{'Run':<26} {'Precision':>10} {'Recall':>10} {'F1':>10} {'FP':>6} {'FN':>6} {'Cost USD':>10}")
for name, items in [('Gemma provider default', baseline), ('Gemma reasoning disabled', results), ('JEV', read_csv(DATA / 'jev_test_predictions.csv'))]:
    s = evaluate(items)
    print(f"{name:<26} {s['precision']:>10.3f} {s['recall']:>10.3f} {s['f1']:>10.3f} {s['fp']:>6} {s['fn']:>6} {s['total_cost_usd']:>10.4f}")
changed = [int(r['source_row']) for r in results if int(r['prediction']) != int(baseline_by_row[int(r['source_row'])]['prediction'])]
print('Changed predictions:', len(changed), changed)


Reasoning token telemetry: 559 / 559
Rows reporting reasoning despite disable request: []
Run                         Precision     Recall         F1     FP     FN   Cost USD
Gemma provider default          0.459      0.918      0.612     79      6     0.0226
Gemma reasoning disabled        0.462      0.904      0.611     77      7     0.0225
JEV                             0.680      0.699      0.689     24     22     0.0185
Changed predictions: 3 [47, 122, 382]


## Completed run — October 1, 2026

All 559 responses reported zero reasoning tokens and no returned reasoning. The run cost $0.0225.

| Run | Precision | Recall | F1 | False positives | False negatives |
|---|---:|---:|---:|---:|---:|
| Gemma, historical provider default | 45.9% | 91.8% | 61.2% | 79 | 6 |
| Gemma, reasoning disabled | 46.2% | 90.4% | 61.1% | 77 | 7 |
| JEV | 68.0% | 69.9% | 68.9% | 24 | 22 |

Only 3 of 559 predictions changed (rows 47, 122, 382): two false positives were removed and one true positive was lost. This run shows little practical change on this sample. Historical reasoning telemetry was not saved, so it does not establish the effect of turning off previously enabled reasoning.
